# Motif-pair feature selection with TabPFN and SHAP

This notebook selects oriented motif pairs that predict boundary pairing and tests them on an independent dataset.

**Data**

- **Dataset 1**: inserted boundary pairs. Each `ID` contains `TADA` or `TADB` (insertion site). Used for preselection, cross-validation (CV), feature selection and threshold tuning.
- **Dataset 2**: genomic boundary pairs (positives and negatives). Used only for the final evaluation.
- **Features**: one column per oriented motif pair (for example `convergent_CISBP.pita_AND_CISBP.pita`), holding its count in the boundary pair.
- **Label**: `Boundary` = 1 (pairing, "working") or 0 (non-pairing).

**Steps**

1. Preselect motif pairs by working rate in dataset 1 and occurrence in the genome.
2. 5-fold CV on dataset 1 with all preselected features. In each fold: filter features, fit TabPFN, compute SHAP values on the held-out fold.
3. Rank features by the median over folds of mean |SHAP|. Keep the top 40.
4. Refit on the same folds with the 40 features. Tune the decision threshold to maximize mean F1.
5. Fit the final model on all of dataset 1. Evaluate it on dataset 2.

Dataset 2 is not used in steps 1 to 4.

**Outputs** (in `OUT_DIR`): `tables/` (selected features, SHAP ranking, metrics, threshold) and `images/` (SHAP, ROC and precision-recall plots).

The same analysis is implemented in `pipeline.py` (run with `python run_analysis.py config/manuscript.yaml`). This notebook is kept for interactive use.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import (
    accuracy_score, auc, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from tabpfn import TabPFNClassifier
from tabpfn_extensions import interpretability

## Parameters

Paths are relative to the `notebooks/` folder. Place the input tables in `data/processed/` (see `data/processed/README.md`).

In [ ]:
# ---- Selection and model ----
TARGET_N_FEATURES = 40   # motif pairs kept after SHAP ranking (step 3)
N_SPLITS = 5             # CV folds on dataset 1
RANDOM_STATE = 42        # seed for CV splits and threshold tuning

# ---- Feature filtering inside each training fold (step 2) ----
VAR_THRESHOLD = 1e-3     # drop features with variance <= this
CORR_THRESHOLD = 0.9     # drop one feature of each pair with |Pearson r| > this

# ---- Preselection thresholds (step 1) ----
min_workA = 0.8            # working rate at the TADA site
min_workB = 0.8            # working rate at the TADB site
min_rep_ins = 5            # working pairs carrying the motif pair, TADA + TADB
min_freq_genome_or = 0.25  # Freq_genome_Or threshold
min_rep_genome_or = 10     # N_genome_Or threshold
min_rep_genome_anyor = 0   # N_genome_anyOr threshold

# ---- GPU ----
# Set to a GPU index (e.g. "3") to choose a GPU on a multi-GPU node.
# None keeps the default GPU.
GPU_ID = None
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
DEVICE = "cuda"          # "cuda" or "cpu"

# ---- Input tables ----
IN_DIR = "../data/processed"
insbound_file = f"{IN_DIR}/ML_inserted_boundarypairs_noLR.csv"            # dataset 1
genomepos_file = f"{IN_DIR}/ML_positiveGenome_boundarypairs_noLR_ext.csv"  # dataset 2, positives
genomeneg_file = f"{IN_DIR}/ML_negativeGenome_boundarypairs_noLR_ext.csv"  # dataset 2, negatives
selfile = f"{IN_DIR}/all_motifpairs_metrics_LRagnostic_reducedMotifs_tandemfixed.csv"  # motif-pair metrics

# ---- Output folder (name encodes the main parameters) ----
name = (f"FEATSEL{TARGET_N_FEATURES}_LRagnostic_work{min_workA}A{min_workB}B"
        f"_minrepins{min_rep_ins}_freqgenor{min_freq_genome_or}"
        f"_minrepgenomeor{min_rep_genome_or}")
OUT_DIR = f"../results/{name}"
os.makedirs(f"{OUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)

## Step 1: preselect motif pairs

The metrics table has one row per oriented motif pair (`Motif_pair_wOr`):

- `N_work_TADA`, `N_work_TADB`: working pairs (`Boundary == 1`) at that insertion site carrying the motif pair.
- `Freq_work_TADA`, `Freq_work_TADB`: working rate, i.e. `N_work` divided by the number of pairs at that site carrying the motif pair.
- `Freq_genome_Or`, `N_genome_Or`, `N_genome_anyOr`: occurrence of the motif pair in the genome.

Working rates are computed from the dataset 1 labels, so this step uses those labels. CV estimates in dataset 1 are therefore optimistic; dataset 2 gives the unbiased estimate.

In [ ]:
motif_metrics = pd.read_csv(selfile)

passes = (
    (motif_metrics["Freq_work_TADA"] >= min_workA)
    & (motif_metrics["Freq_work_TADB"] >= min_workB)
    & ((motif_metrics["N_work_TADA"] + motif_metrics["N_work_TADB"]) >= min_rep_ins)
    & (motif_metrics["Freq_genome_Or"] >= min_freq_genome_or)
    & (motif_metrics["N_genome_Or"] >= min_rep_genome_or)
    & (motif_metrics["N_genome_anyOr"] >= min_rep_genome_anyor)
)
selmotifs = motif_metrics.loc[passes, "Motif_pair_wOr"].tolist()
print(f"{len(selmotifs)} motif pairs pass preselection")

## Load the datasets

Both tables have columns `ID`, `Boundary` and one column per motif pair. Only preselected motif pairs present in dataset 1 are kept, in both datasets.

In [ ]:
data_cv = pd.read_csv(insbound_file)                     # dataset 1
data_ext = pd.concat([pd.read_csv(genomepos_file),       # dataset 2
                      pd.read_csv(genomeneg_file)], ignore_index=True)

X_cv = data_cv.drop(columns=["ID", "Boundary"])
y_cv = data_cv["Boundary"].values
X_ext = data_ext.drop(columns=["ID", "Boundary"])
y_ext = data_ext["Boundary"].values

selmotifs = X_cv.columns.intersection(selmotifs)
X_cv = X_cv[selmotifs]
X_ext = X_ext[selmotifs]

print(f"Dataset 1: {X_cv.shape[0]} pairs ({y_cv.mean():.0%} positive)")
print(f"Dataset 2: {X_ext.shape[0]} pairs ({y_ext.mean():.0%} positive)")
print(f"Features: {X_cv.shape[1]} motif pairs")

## Helper: feature filtering

Applied inside each CV training fold (step 2). Thresholds are computed on the training fold only; the same columns are then removed from the held-out fold, so no information leaks from the held-out fold.

In [ ]:
def filter_features(X_train, X_test, var_threshold, corr_threshold, fold, out_dir):
    """Remove low-variance and highly correlated features.

    1. Drop features with variance <= var_threshold in the training fold.
    2. For each pair of remaining features with |Pearson r| > corr_threshold
       in the training fold, drop the later column.
    The same columns are dropped from the test fold. Removed correlated
    features are saved to tables/dropped_features_correlation_fold<fold>.csv.

    Returns: X_train, X_test (filtered), dict of dropped features,
    table of removed correlated pairs.
    """
    # 1. Low variance
    variances = X_train.var(axis=0)
    low_var_feats = variances[variances <= var_threshold].index.tolist()
    X_train = X_train.drop(columns=low_var_feats)
    X_test = X_test.drop(columns=low_var_feats, errors="ignore")

    # 2. High correlation (upper triangle, so each pair is checked once)
    corr = X_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    high_corr_feats, corr_records = [], []
    for col in upper.columns:
        high_corr = upper[col][upper[col] > corr_threshold]
        if not high_corr.empty:
            high_corr_feats.append(col)
            for kept_col, corr_val in high_corr.items():
                corr_records.append({"kept_feature": kept_col,
                                     "removed_feature": col,
                                     "correlation": corr_val})
    X_train = X_train.drop(columns=high_corr_feats)
    X_test = X_test.drop(columns=high_corr_feats, errors="ignore")

    correlation_table = pd.DataFrame(corr_records)
    correlation_table.to_csv(
        f"{out_dir}/tables/dropped_features_correlation_fold{fold}.csv", index=False)

    dropped = {"low_variance": low_var_feats, "high_correlation": high_corr_feats}
    return X_train, X_test, dropped, correlation_table

## Step 2: cross-validation with all preselected features

For each of the 5 stratified folds of dataset 1:

1. Filter features on the training fold.
2. Fit TabPFN (`balance_probabilities=True` corrects for class imbalance).
3. Record metrics on the held-out fold at threshold 0.5.
4. Compute SHAP values for class 1 on the held-out fold (permutation explainer, budget of 3 evaluations per feature). Store the mean |SHAP| and mean signed SHAP of each feature. Features removed by the filter in that fold get 0.

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

results = []                     # per-fold metrics and predictions
shap_importance_all_folds = []   # mean |SHAP| per feature, one Series per fold
shap_signed_all_folds = []       # mean signed SHAP per feature, one Series per fold

for fold, (train_idx, test_idx) in enumerate(skf.split(X_cv, y_cv), 1):
    print(f"\n===== Fold {fold} =====")
    X_train, X_test = X_cv.iloc[train_idx], X_cv.iloc[test_idx]
    y_train, y_test = y_cv[train_idx], y_cv[test_idx]

    X_train, X_test, dropped_features, _ = filter_features(
        X_train, X_test, VAR_THRESHOLD, CORR_THRESHOLD, fold, OUT_DIR)

    model = TabPFNClassifier(device=DEVICE, balance_probabilities=True)
    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

    # SHAP values for class 1: array (samples x features)
    shap_values = interpretability.shap.get_shap_values(
        model, X_test, max_evals=3 * X_test.shape[1])
    shap_vals = shap_values.values[:, :, 1]

    shap_importance_all_folds.append(
        pd.Series(np.abs(shap_vals).mean(axis=0), index=X_test.columns)
        .reindex(X_cv.columns, fill_value=0.0))
    shap_signed_all_folds.append(
        pd.Series(shap_vals.mean(axis=0), index=X_test.columns)
        .reindex(X_cv.columns, fill_value=0.0))

    results.append({"fold": fold, "train_idx": train_idx, "test_idx": test_idx,
                    "metrics": metrics, "y_prob": y_prob, "y_pred": y_pred})

## Step 3: rank by SHAP and select the top 40

Features are ranked by the median over folds of mean |SHAP|. The median limits the influence of a single fold. The number of features is fixed at `TARGET_N_FEATURES`; dataset 2 is not used.

In [ ]:
shap_importance_df = pd.concat(shap_importance_all_folds, axis=1)
median_shap = shap_importance_df.median(axis=1).sort_values(ascending=False)
median_shap.rename("median_abs_shap").to_csv(f"{OUT_DIR}/tables/shap_ranking_cv.csv")

selected_features = median_shap.index[:TARGET_N_FEATURES].tolist()
print(f"Selected {len(selected_features)} features")
pd.Series(selected_features).to_csv(f"{OUT_DIR}/tables/selected_features.csv", index=False)

## Step 4: cross-validation with the selected features and threshold tuning

The model is refit on the same folds using only the selected features, without further filtering. The out-of-fold probabilities are then used to choose the decision threshold that maximizes mean F1 across folds (Optuna, 50 trials, range 0.05 to 0.95).

The features and the threshold were both chosen on these folds, so the CV metrics below are optimistic.

In [ ]:
results_redmodel = []   # out-of-fold probabilities with the selected features

for fold, (train_idx, test_idx) in enumerate(skf.split(X_cv, y_cv), 1):
    model = TabPFNClassifier(device=DEVICE, balance_probabilities=True)
    model.fit(X_cv.iloc[train_idx][selected_features].values, y_cv[train_idx])
    y_prob = model.predict_proba(X_cv.iloc[test_idx][selected_features].values)[:, 1]
    results_redmodel.append({"fold": fold, "train_idx": train_idx,
                             "test_idx": test_idx, "y_prob": y_prob})

In [ ]:
def objective(trial):
    """Mean F1 across folds for a candidate threshold."""
    threshold = trial.suggest_float("threshold", 0.05, 0.95)
    f1s = [f1_score(y_cv[r["test_idx"]], (r["y_prob"] >= threshold).astype(int),
                    zero_division=0)
           for r in results_redmodel]
    return np.mean(f1s)


# Seeded sampler so that the threshold is reproducible
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_threshold = study.best_params["threshold"]
print(f"Best threshold = {best_threshold:.3f}")
print(f"Best CV F1     = {study.best_value:.3f}")
pd.Series(study.best_params).to_csv(f"{OUT_DIR}/tables/optuna_best_threshold.csv")

In [ ]:
# CV metrics of the selected-feature model at the tuned threshold
cv_metrics_opt = []
for r in results_redmodel:
    y_true = y_cv[r["test_idx"]]
    y_pred = (r["y_prob"] >= best_threshold).astype(int)
    cv_metrics_opt.append({
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, r["y_prob"]),
    })
pd.DataFrame(cv_metrics_opt).mean()

## Step 5: final model and evaluation on dataset 2

The final model is fit on all of dataset 1 with the selected features and applied to dataset 2 with the tuned threshold. This is the unbiased performance estimate.

In [ ]:
final_model = TabPFNClassifier(device=DEVICE, balance_probabilities=True)
final_model.fit(X_cv[selected_features].values, y_cv)

y_ext_prob = final_model.predict_proba(X_ext[selected_features].values)[:, 1]
y_ext_pred = (y_ext_prob >= best_threshold).astype(int)

external_metrics = {
    "accuracy": accuracy_score(y_ext, y_ext_pred),
    "precision": precision_score(y_ext, y_ext_pred, zero_division=0),
    "recall": recall_score(y_ext, y_ext_pred, zero_division=0),
    "f1": f1_score(y_ext, y_ext_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_ext, y_ext_prob),
}
external_metrics

## Save metrics

`metrics.csv` rows:

- `1` to `5`: step 2 folds (all preselected features, threshold 0.5).
- `cv_opt_1` to `cv_opt_5`: step 4 folds (selected features, tuned threshold).
- `external`: final model on dataset 2 (tuned threshold).

In [ ]:
metrics_df = pd.DataFrame(
    [{"fold": r["fold"], **r["metrics"]} for r in results]
    + [{"fold": f"cv_opt_{i + 1}", **m} for i, m in enumerate(cv_metrics_opt)]
    + [{"fold": "external", **external_metrics}]
)
metrics_df.to_csv(f"{OUT_DIR}/tables/metrics.csv", index=False)
metrics_df

## Figures

### SHAP across CV folds

Median over folds of the mean signed SHAP value (class 1). Red: higher counts push towards pairing; blue: towards non-pairing. The 10 most positive and 10 most negative motif pairs are shown.

In [ ]:
median_signed = pd.concat(shap_signed_all_folds, axis=1).median(axis=1)

top_pos = median_signed.sort_values(ascending=False).head(10)
top_neg = median_signed.sort_values().head(10)
shap_plot_df = pd.concat([top_pos, top_neg]).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(shap_plot_df.index, shap_plot_df.values,
         color=["red" if v > 0 else "blue" for v in shap_plot_df.values])
plt.axvline(0, color="black", lw=0.8)
plt.gca().invert_yaxis()
plt.xlabel("Median SHAP value (class 1)")
plt.title("Top positive and negative SHAP features (CV)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/images/SHAP_top_posneg.png", dpi=300)
plt.show()

### SHAP of the final model

Mean signed SHAP value over all of dataset 1 for the final model (budget of 10 evaluations per feature). This takes a few minutes on a GPU.

In [ ]:
shap_vals_final = interpretability.shap.get_shap_values(
    final_model, X_cv[selected_features], max_evals=10 * len(selected_features))

shap_signed = pd.Series(shap_vals_final.values[:, :, 1].mean(axis=0),
                        index=selected_features)
shap_signed.rename("mean_shap").to_csv(f"{OUT_DIR}/tables/shap_final_model.csv")

top_pos = shap_signed.sort_values(ascending=False).head(10)
top_neg = shap_signed.sort_values().head(10)
shap_plot_df = pd.concat([top_pos, top_neg]).sort_values()

plt.figure(figsize=(8, 6))
plt.barh(shap_plot_df.index, shap_plot_df.values,
         color=["red" if v > 0 else "blue" for v in shap_plot_df.values])
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Mean SHAP value (class 1)")
plt.title("Final model SHAP: top positive and negative features")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/images/SHAP_final_model_top_posneg.png", dpi=300)
plt.show()

### ROC curves

Blue: mean over CV folds of the selected-feature model, with a band of 1 SD. Red: final model on dataset 2.

In [ ]:
mean_fpr = np.linspace(0, 1, 100)
tprs = []
for r in results_redmodel:
    fpr, tpr, _ = roc_curve(y_cv[r["test_idx"]], r["y_prob"])
    tprs.append(np.interp(mean_fpr, fpr, tpr))
mean_tpr, std_tpr = np.mean(tprs, axis=0), np.std(tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)

fpr_ext, tpr_ext, _ = roc_curve(y_ext, y_ext_prob)
ext_auc = auc(fpr_ext, tpr_ext)

plt.figure(figsize=(10, 6))
plt.plot(mean_fpr, mean_tpr, label=f"Mean CV ROC (AUC = {mean_auc:.2f})")
plt.fill_between(mean_fpr, np.maximum(0, mean_tpr - std_tpr),
                 np.minimum(1, mean_tpr + std_tpr), alpha=0.2)
plt.plot(fpr_ext, tpr_ext, color="red", label=f"External ROC (AUC = {ext_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=2, label="Random classifier")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig(f"{OUT_DIR}/images/ROC_CV_vs_External.png", dpi=300)
plt.show()

### Precision-recall curves

Same layout as the ROC plot. Dashed lines show the fraction of positives in each dataset, which is the precision of a random classifier.

In [ ]:
recall_grid = np.linspace(0, 1, 100)
precisions = []
for r in results_redmodel:
    precision, recall, _ = precision_recall_curve(y_cv[r["test_idx"]], r["y_prob"])
    order = np.argsort(recall)          # np.interp needs increasing x
    precisions.append(np.interp(recall_grid, recall[order], precision[order]))
mean_precision, std_precision = np.mean(precisions, axis=0), np.std(precisions, axis=0)
cv_aupr = auc(recall_grid, mean_precision)

precision_ext, recall_ext, _ = precision_recall_curve(y_ext, y_ext_prob)
ext_aupr = auc(recall_ext, precision_ext)

plt.figure(figsize=(10, 6))
plt.plot(recall_grid, mean_precision, label=f"Mean CV PR (AUPR = {cv_aupr:.2f})")
plt.fill_between(recall_grid, np.maximum(0, mean_precision - std_precision),
                 np.minimum(1, mean_precision + std_precision), alpha=0.2)
plt.plot(recall_ext, precision_ext, color="red", label=f"External PR (AUPR = {ext_aupr:.2f})")
plt.plot([0, 1], [np.mean(y_cv)] * 2, color="blue", linestyle="--", lw=2, label="CV baseline")
plt.plot([0, 1], [np.mean(y_ext)] * 2, color="red", linestyle="--", lw=2, label="External baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig(f"{OUT_DIR}/images/PR_CV_vs_External.png", dpi=300)
plt.show()